In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Load credentials from .env instead of hardcoding them 
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# SQLAlchemy's "engine" is the reusable connection object pandas uses under the hood
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [46]:
model_df = pd.read_sql("SELECT * FROM order_modeling_dataset;", engine)
print(model_df.shape)
model_df.head()

(99441, 14)


,order_id,review_score,delivery_days,handling_days,transit_days,is_late,order_value,freight_value,item_count,avg_distance_km,max_distance_km,customer_state,primary_category,order_status
0,00018f77f2f0320c557190d7a144bdd3,4.0,16.0,8.0,8.0,False,239.90,19.93,1.0,585.6,585.6,SP,pet_shop,delivered
1,000229ec398224ef6ca0657da4fc703e,5.0,7.0,1.0,6.0,False,199.00,17.87,1.0,312.3,312.3,MG,furniture_decor,delivered
2,00048cc3ae777c65dbb7d2a0634bc1ea,4.0,6.0,1.0,5.0,False,21.90,12.69,1.0,161.9,161.9,MG,housewares,delivered
3,0005a1a1728c9d785b8e2b08b904576c,1.0,9.0,8.0,1.0,True,145.95,11.65,1.0,53.2,53.2,SP,health_beauty,delivered
4,00061f2a7bc09da83e415a52dc8a4af1,5.0,4.0,2.0,1.0,False,59.99,8.88,1.0,154.1,154.1,SP,health_beauty,delivered


In [ ]:
# PHASE 4 QUASI EXPERIMENTAL CAUSAL ANALYSIS
 
model_df = model_df[~model_df["order_status"].isin(["canceled", "unavailable"])]
model_df = model_df[model_df["review_score"].notna()]
model_df = model_df[model_df["delivery_days"].notna()]
model_df = model_df[model_df["avg_distance_km"].notna()]

print(model_df.shape)
print(model_df["is_late"].isna().sum())  # should be 0 now
 
print("Final modeling dataset shape:", model_df.shape)

# Quick check: did any of these filters dramatically shift the review score distribution?
# If filtering changed the mix a lot, that's worth noting alongside the MNAR limitation.
model_df["review_score"].value_counts(normalize=True).sort_index() * 100

(95349, 14)
0
Final modeling dataset shape: (95349, 14)


review_score
1.0     9.764130
2.0     3.048800
3.0     8.258083
4.0    19.715991
5.0    59.212996
Name: proportion, dtype: float64

In [ ]:
 

# ## Formal multicollinearity check (VIF)
# Correlation (Phase 3) only looks at pairs of variables. VIF looks at each variable
# against ALL other predictors combined — a more rigorous test before trusting
# individual regression coefficients.
 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


 
vif_df = model_df[["delivery_days", "avg_distance_km", "order_value", "freight_value", "item_count"]].copy()
vif_df["is_late"] = model_df["is_late"].astype(int)
vif_df = add_constant(vif_df)
 
vif_results = pd.DataFrame()
vif_results["feature"] = vif_df.columns
vif_results["VIF"] = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
vif_results[vif_results["feature"] != "const"]  

,feature,VIF
1,delivery_days,1.901317
2,avg_distance_km,1.373615
3,order_value,1.208066
4,freight_value,1.670506
5,item_count,1.283563
6,is_late,1.605322


Outcome: Variance Inflation Factor well below the 5.0 threshold. Safe to say no multicollinearity -> delivery_days and is_late can go into the regression as independent predictors

In [50]:
# %%
# Standardizing puts all continuous variables on the same scale (mean 0, std 1),
# which is a common fix for optimizer convergence problems — it doesn't change
# WHAT the model finds, just makes it numerically easier to find it.
continuous_vars = ["delivery_days", "avg_distance_km", "order_value", "freight_value", "item_count"]
 
for col in continuous_vars:
    model_df[col + "_z"] = (model_df[col] - model_df[col].mean()) / model_df[col].std()
 
formula = (
    "review_score_cat ~ delivery_days_z + is_late_num + avg_distance_km_z "
    "+ order_value_z + freight_value_z + item_count_z "
    "+ C(category_grouped) + C(customer_state)"
)
 
ordinal_model = OrderedModel.from_formula(formula, data=model_df, distr="logit")
ordinal_result = ordinal_model.fit(method="bfgs", maxiter=1000)
 
# %%
# Check convergence explicitly rather than scrolling for the warning
print("Converged:", ordinal_result.mle_retvals.get("converged", "unknown"))
 
# %%
# Pull ONLY the rows we actually care about, avoiding the truncated-table problem
summary_df = pd.DataFrame({
    "coef": ordinal_result.params,
    "std_err": ordinal_result.bse,
    "p_value": ordinal_result.pvalues,
})
 
key_vars = ["delivery_days_z", "is_late_num", "avg_distance_km_z", "order_value_z", "freight_value_z", "item_count_z"]
summary_df.loc[key_vars]

Optimization terminated successfully.
         Current function value: 1.099929
         Iterations: 187
         Function evaluations: 188
         Gradient evaluations: 188
Converged: True


,coef,std_err,p_value
delivery_days_z,-0.508606,0.010441,0.000000e+00
is_late_num,-1.299755,0.029862,0.000000e+00
avg_distance_km_z,0.034340,0.013390,1.032780e-02
order_value_z,0.009705,0.007438,1.919542e-01
freight_value_z,-0.008538,0.008866,3.355716e-01
item_count_z,-0.224128,0.007750,7.054749e-184


delivery_days_z coefficient -0.51 pval << 0.001 highly significant. 
is_late_num coefficient -1.30 pval << 0.001 highly significant.
Reason to believe they are strong predictors of satisfaction holding all others constant

order_value_z and freight_value_z are not statistically significant

item_count_z coefficient -1.23 pval << 0.001 highly significant -> more items in order predicts lower satisfaction (more risk of damaged item, missing, packacging issues) regardless of speed.

avg_distance_km_z coefficient +0.03 pval = 0.010 significant but barely, further away orders correlate marginially better reviews once delivery time and lateness are accounted for. 

In [51]:
# ## Translating coefficients into plain-English effect sizes
 
# %%
import numpy as np
 
# Odds ratios: exp(coefficient). An odds ratio of 0.6 means "40% lower odds of a
# higher review category"; 1.3 means "30% higher odds."
summary_df["odds_ratio"] = np.exp(summary_df["coef"])
summary_df.loc[key_vars, ["coef", "odds_ratio", "p_value"]]
 
# %%
# Convert delivery_days back from standardized units to real days, since
# "-0.51 per standard deviation" isn't something you can say in an interview.
delivery_days_std = model_df["delivery_days"].std()
coef_per_day = ordinal_result.params["delivery_days_z"] / delivery_days_std
odds_ratio_per_day = np.exp(coef_per_day)
 
print(f"Effect per additional delivery day: coefficient = {coef_per_day:.4f}")
print(f"Odds ratio per additional day: {odds_ratio_per_day:.4f}")
print(f"--> Each extra day of delivery time is associated with a {(1 - odds_ratio_per_day) * 100:.1f}% "
      f"reduction in the odds of a higher review score, holding all else constant.")
 
# %%
# Same translation for is_late (already binary, no conversion needed — this is just the headline number)
is_late_odds_ratio = np.exp(ordinal_result.params["is_late_num"])
print(f"Odds ratio for a LATE order: {is_late_odds_ratio:.4f}")
print(f"--> A late order has {(1 - is_late_odds_ratio) * 100:.1f}% lower odds of a higher review score "
      f"than an on-time order, holding delivery duration and all other factors constant.")

Effect per additional delivery day: coefficient = -0.0537
Odds ratio per additional day: 0.9477
--> Each extra day of delivery time is associated with a 5.2% reduction in the odds of a higher review score, holding all else constant.
Odds ratio for a LATE order: 0.2726
--> A late order has 72.7% lower odds of a higher review score than an on-time order, holding delivery duration and all other factors constant.


In [52]:
# ## Stratified check: does delivery speed's effect vary by category? (H4)
# Adding an interaction term (delivery_days_z * category) tests whether the SLOPE of
# delivery's effect differs by category — more rigorous than comparing separate
# averages per category, since it's tested within one unified model with a p-value
# attached to each difference.
 
# %%
formula_interact = (
    "review_score_cat ~ delivery_days_z * C(category_grouped) + is_late_num + avg_distance_km_z "
    "+ order_value_z + freight_value_z + item_count_z + C(customer_state)"
)
 
interact_model = OrderedModel.from_formula(formula_interact, data=model_df, distr="logit")
interact_result = interact_model.fit(method="bfgs", maxiter=1000)
print("Converged:", interact_result.mle_retvals.get("converged", "unknown"))
 
# %%
# Pull ONLY the interaction terms — these tell us whether each category's
# delivery-sensitivity differs significantly from the reference ("other") category.
interact_summary = pd.DataFrame({
    "coef": interact_result.params,
    "std_err": interact_result.bse,
    "p_value": interact_result.pvalues,
})
 
interaction_rows = interact_summary[interact_summary.index.str.contains("delivery_days_z:")]
interaction_rows.sort_values("p_value")

Optimization terminated successfully.
         Current function value: 1.099649
         Iterations: 198
         Function evaluations: 199
         Gradient evaluations: 199
Converged: True


,coef,std_err,p_value
delivery_days_z:C(category_grouped)[T.toys],-0.136780,0.051511,0.007923
delivery_days_z:C(category_grouped)[T.health_beauty],-0.089259,0.041814,0.032790
delivery_days_z:C(category_grouped)[T.sports_leisure],-0.092217,0.043739,0.035002
delivery_days_z:C(category_grouped)[T.baby],-0.081500,0.053686,0.128989
delivery_days_z:C(category_grouped)[T.cool_stuff],0.077064,0.051501,0.134560
delivery_days_z:C(category_grouped)[T.telephony],0.045259,0.048679,0.352494
delivery_days_z:C(category_grouped)[T.watches_gifts],0.039919,0.044920,0.374189
delivery_days_z:C(category_grouped)[T.other],-0.033297,0.037837,0.378847
delivery_days_z:C(category_grouped)[T.bed_bath_table],0.027523,0.041435,0.506538
delivery_days_z:C(category_grouped)[T.electronics],0.031580,0.056552,0.576558


significant: toys, health_beauty, sports_leisure

*** MULTIPLE COMPARISON PROBLEM Roughly 0.75 false positives by pure chance even if delivery effect identical
Bonferroni Correction: none of the significant categories survive it.

Do not have strong evidence to conclude specific categories has meaningfully more delivery sensitive information than others. 